# Overnight Full-Context Inference Runner

This notebook runs `select_slices_and_predict.py` on the remaining volumes with full 3D context.

Execution model:
- Local GPU in VS Code
- Sequential per-volume execution (safe for overnight)
- Logs per sample + master log
- Resume behavior by skipping samples with existing final annotation TIFF

## 1) Install And Environment Setup

Run this only if packages are missing in your active kernel.

In [1]:
INSTALL_MISSING = False

if INSTALL_MISSING:
    # Keep this short and explicit for local environment use.
    %pip install nnunetv2 nibabel tifffile scikit-image

In [2]:
import importlib
import platform
import sys

print(f"Python executable: {sys.executable}")
print(f"Python version: {platform.python_version()}")

for pkg in ["nnunetv2", "nibabel", "tifffile", "skimage", "numpy"]:
    mod = importlib.import_module(pkg)
    version = getattr(mod, "__version__", "unknown")
    print(f"{pkg}: {version}")

Python executable: c:\Users\rony.schwartz\.conda\envs\venv-napari\python.exe
Python version: 3.11.15
nnunetv2: unknown
nibabel: 5.4.2
tifffile: 2026.3.3
skimage: 0.26.0
numpy: 2.4.3


## 2) Imports And Base Setup

Imports, reproducibility seed, and base path constants.

In [3]:
from pathlib import Path
from datetime import datetime
import csv
import json
import os
import random
import subprocess
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PYTHON_EXE = r"C:/Users/rony.schwartz/.conda/envs/venv-napari/python.exe"
SCRIPT = Path(r"C:/Users/rony.schwartz/Documents/nnUNet4SoilXrayCT/select_slices_and_predict.py")
INPUT_DIR = Path(r"//hive3065/Yael_Mishael/Rony/remote_computer backup/10.5")
MODEL = Path(r"C:/Users/rony.schwartz/Documents/nnUNet_resources/bnei_reem/nnUNet_results/Dataset777_GCEF/nnUNetTrainer_betterIgnoreSampling__nnUNetPlans__3d_fullres")
OUTPUT_ROOT = Path(r"//hive3065/Yael_Mishael/Rony/remote_computer backup/10.5/bootstrap_outputs_fullctx")

print(f"Seed: {SEED}")
print(f"Python executable for runs: {PYTHON_EXE}")

Seed: 42
Python executable for runs: C:/Users/rony.schwartz/.conda/envs/venv-napari/python.exe


## 3) Run Parameters

Edit only this cell when you want to change which volumes to process.

In [4]:
VOLUMES = [
    "mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif",
    "mishmar_hanegev_Cu011_samp_3_Rec_nlm.tif",
    "nlm_volume.tif",
]

CHECKPOINT_NAME = "checkpoint_final.pth"
RETRY_ON_FAIL = 1

MASTER_LOG = OUTPUT_ROOT / "overnight_master.log"
SUMMARY_JSON = OUTPUT_ROOT / "overnight_summary.json"
SUMMARY_CSV = OUTPUT_ROOT / "overnight_summary.csv"

print("Configured volumes:")
for name in VOLUMES:
    print(f"- {name}")

Configured volumes:
- mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
- mishmar_hanegev_Cu011_samp_3_Rec_nlm.tif
- nlm_volume.tif


## 4) Core Functions

Functions for validation, pending detection, single-sample run, and orchestration.

In [5]:
def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def append_master_log(message: str) -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    with MASTER_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{now_iso()}] {message}\n")


def validate_paths() -> None:
    checks = {
        "PYTHON_EXE": Path(PYTHON_EXE).exists(),
        "SCRIPT": SCRIPT.exists(),
        "INPUT_DIR": INPUT_DIR.exists(),
        "MODEL": MODEL.exists(),
    }
    for key, ok in checks.items():
        print(f"{key}: {'OK' if ok else 'MISSING'}")
    missing = [k for k, ok in checks.items() if not ok]
    if missing:
        raise FileNotFoundError(f"Missing required paths: {missing}")


def sample_id_from_name(volume_name: str) -> str:
    return Path(volume_name).stem


def annotation_path_for_sample(sample_id: str) -> Path:
    return OUTPUT_ROOT / sample_id / "annotations" / f"{sample_id}.tif"


def split_pending_done(volume_names: list[str]) -> tuple[list[str], list[str]]:
    pending, done = [], []
    for volume_name in volume_names:
        sample_id = sample_id_from_name(volume_name)
        ann_path = annotation_path_for_sample(sample_id)
        if ann_path.exists():
            done.append(volume_name)
        else:
            pending.append(volume_name)
    return pending, done


def run_one_volume(volume_name: str, retry_on_fail: int = 1) -> dict:
    sample_id = sample_id_from_name(volume_name)
    input_volume = INPUT_DIR / volume_name
    out_dir = OUTPUT_ROOT / sample_id
    meta_dir = out_dir / "metadata"
    meta_dir.mkdir(parents=True, exist_ok=True)

    stdout_log = meta_dir / "overnight_stdout.log"
    stderr_log = meta_dir / "overnight_stderr.log"

    cmd = [
        PYTHON_EXE,
        str(SCRIPT),
        "--input_volume", str(input_volume),
        "--model_or_checkpoint", str(MODEL),
        "--output_dir", str(out_dir),
        "--checkpoint_name", CHECKPOINT_NAME,
    ]

    attempts = 1 + max(0, retry_on_fail)
    last_code = None

    for attempt in range(1, attempts + 1):
        append_master_log(f"START sample={sample_id} attempt={attempt}")
        print(f"START  sample={sample_id} attempt={attempt}")

        with stdout_log.open("a", encoding="utf-8") as out_f, stderr_log.open("a", encoding="utf-8") as err_f:
            out_f.write(f"\n===== {now_iso()} attempt={attempt} cmd={' '.join(cmd)} =====\n")
            err_f.write(f"\n===== {now_iso()} attempt={attempt} cmd={' '.join(cmd)} =====\n")
            proc = subprocess.run(cmd, stdout=out_f, stderr=err_f, text=True)
            last_code = proc.returncode

        if last_code == 0:
            append_master_log(f"DONE sample={sample_id} attempt={attempt}")
            print(f"DONE   sample={sample_id}")
            break

        append_master_log(f"FAIL sample={sample_id} attempt={attempt} exit={last_code}")
        print(f"FAIL   sample={sample_id} attempt={attempt} exit={last_code}")

    final_ann = annotation_path_for_sample(sample_id)
    status = "DONE" if last_code == 0 and final_ann.exists() else "FINAL_FAIL"
    if status == "FINAL_FAIL":
        append_master_log(f"FINAL_FAIL sample={sample_id} exit={last_code}")

    return {
        "sample_id": sample_id,
        "volume_name": volume_name,
        "status": status,
        "exit_code": int(last_code) if last_code is not None else -1,
        "annotation_path": str(final_ann),
        "annotation_exists": final_ann.exists(),
    }


def save_summary(results: list[dict]) -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    with SUMMARY_JSON.open("w", encoding="utf-8") as f:
        json.dump({"timestamp": now_iso(), "results": results}, f, indent=2)

    with SUMMARY_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["sample_id", "volume_name", "status", "exit_code", "annotation_path", "annotation_exists"],
        )
        writer.writeheader()
        writer.writerows(results)

    print(f"Saved summary JSON: {SUMMARY_JSON}")
    print(f"Saved summary CSV : {SUMMARY_CSV}")


def tail_master_log(n: int = 100) -> None:
    if not MASTER_LOG.exists():
        print(f"Master log not found yet: {MASTER_LOG}")
        return
    lines = MASTER_LOG.read_text(encoding="utf-8").splitlines()
    print("\n".join(lines[-n:]))

In [6]:
validate_paths()
pending, done = split_pending_done(VOLUMES)

print("\nAlready done:")
for v in done:
    print(f"- {v}")

print("\nPending:")
for v in pending:
    print(f"- {v}")

PYTHON_EXE: OK
SCRIPT: OK
INPUT_DIR: OK
MODEL: OK

Already done:

Pending:
- mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
- mishmar_hanegev_Cu011_samp_3_Rec_nlm.tif
- nlm_volume.tif


## 5) End-to-End Run In One Cell

Run this cell and let it work overnight.

In [7]:
def main() -> list[dict]:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    append_master_log("Overnight notebook run started")

    validate_paths()
    pending, done = split_pending_done(VOLUMES)

    append_master_log(f"Already done: {len(done)}")
    append_master_log(f"Pending: {len(pending)}")

    results: list[dict] = []

    for v in done:
        sample_id = sample_id_from_name(v)
        ann = annotation_path_for_sample(sample_id)
        results.append(
            {
                "sample_id": sample_id,
                "volume_name": v,
                "status": "SKIPPED_ALREADY_DONE",
                "exit_code": 0,
                "annotation_path": str(ann),
                "annotation_exists": ann.exists(),
            }
        )

    for v in pending:
        results.append(run_one_volume(v, retry_on_fail=RETRY_ON_FAIL))

    save_summary(results)
    append_master_log("Overnight notebook run finished")
    return results


results = main()
print("\nRun complete. Result statuses:")
for r in results:
    print(f"- {r['sample_id']}: {r['status']} (exit={r['exit_code']})")

PYTHON_EXE: OK
SCRIPT: OK
INPUT_DIR: OK
MODEL: OK
START  sample=mishmar_hanegev_Cu011_samp_2_Rec_nlm attempt=1
DONE   sample=mishmar_hanegev_Cu011_samp_2_Rec_nlm
START  sample=mishmar_hanegev_Cu011_samp_3_Rec_nlm attempt=1
DONE   sample=mishmar_hanegev_Cu011_samp_3_Rec_nlm
START  sample=nlm_volume attempt=1
DONE   sample=nlm_volume
Saved summary JSON: \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\bootstrap_outputs_fullctx\overnight_summary.json
Saved summary CSV : \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\bootstrap_outputs_fullctx\overnight_summary.csv

Run complete. Result statuses:
- mishmar_hanegev_Cu011_samp_2_Rec_nlm: DONE (exit=0)
- mishmar_hanegev_Cu011_samp_3_Rec_nlm: DONE (exit=0)
- nlm_volume: DONE (exit=0)


## 6) Quick Sanity Checks

Basic assertions to verify structure and completion quality.

In [8]:
assert isinstance(results, list), "results must be a list"
assert len(results) == len(VOLUMES), "results length must match configured volume list"

for item in results:
    assert item["sample_id"], "sample_id must not be empty"
    assert item["status"] in {"DONE", "FINAL_FAIL", "SKIPPED_ALREADY_DONE"}, "unexpected status"
    if item["status"] in {"DONE", "SKIPPED_ALREADY_DONE"}:
        assert item["annotation_exists"], f"missing annotation for {item['sample_id']}"

print("Sanity checks passed.")

Sanity checks passed.


## 7) Export And Save Results

Tail master log and print final annotation output summary.

In [9]:
tail_master_log(100)

print("\nFinal annotation outputs:")
for vol in VOLUMES:
    sample_id = sample_id_from_name(vol)
    ann = annotation_path_for_sample(sample_id)
    if ann.exists():
        print(f"- {sample_id}: OK ({ann.stat().st_size} bytes) -> {ann}")
    else:
        print(f"- {sample_id}: MISSING -> {ann}")

print(f"\nSummary JSON: {SUMMARY_JSON}")
print(f"Summary CSV : {SUMMARY_CSV}")

[2026-05-10T22:28:45] Overnight notebook run started
[2026-05-10T22:28:45] Already done: 0
[2026-05-10T22:28:45] Pending: 3
[2026-05-10T22:28:45] START sample=mishmar_hanegev_Cu011_samp_2_Rec_nlm attempt=1
[2026-05-10T22:35:45] DONE sample=mishmar_hanegev_Cu011_samp_2_Rec_nlm attempt=1
[2026-05-10T22:35:45] START sample=mishmar_hanegev_Cu011_samp_3_Rec_nlm attempt=1
[2026-05-10T22:42:34] DONE sample=mishmar_hanegev_Cu011_samp_3_Rec_nlm attempt=1
[2026-05-10T22:42:34] START sample=nlm_volume attempt=1
[2026-05-10T22:49:44] DONE sample=nlm_volume attempt=1
[2026-05-10T22:49:44] Overnight notebook run finished

Final annotation outputs:
- mishmar_hanegev_Cu011_samp_2_Rec_nlm: OK (347431542 bytes) -> \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\bootstrap_outputs_fullctx\mishmar_hanegev_Cu011_samp_2_Rec_nlm\annotations\mishmar_hanegev_Cu011_samp_2_Rec_nlm.tif
- mishmar_hanegev_Cu011_samp_3_Rec_nlm: OK (263321008 bytes) -> \\hive3065\Yael_Mishael\Rony\remote_computer backup\10.5\